In [ ]:
import os

GOOGLE_API_KEY= os.environ["GOOGLE_API_KEY"]


os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
import os
from google import genai
from google.genai import types



client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-2.5-pro"  # or "gemini-2.5-pro"

def generate_youtube_script(text: str) -> str:
    """
    Generate a professional YouTube script (with a title) using Gemini.
    No fixed token limit — model will decide response length automatically.
    """
    prompt = f"""
You are a professional YouTube scriptwriter.

Write a complete YouTube video script from the following text.

Rules:
- Start with a catchy TITLE on its own line.
- Then write the SCRIPT in clear, engaging, conversational English.
- Length: as appropriate for the input (usually 2-5 minutes).
- Do not include JSON or code.
- Write it like a real narrator would speak.
- Do not include Musical tones or animation suggestions.
- Do not include anything in parentheses.

Text:
{text.strip()}
"""

    # Notice: no max_output_tokens here
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.4,
        ),
    )

    # Robust extraction logic for all SDK variants
    output_text = ""

    if hasattr(response, "text") and response.text:
        output_text = response.text

    elif hasattr(response, "candidates") and response.candidates:
        candidate = response.candidates[0]
        if hasattr(candidate, "content") and hasattr(candidate.content, "parts"):
            for part in candidate.content.parts:
                if hasattr(part, "text") and part.text:
                    output_text += part.text + "\n"

    elif hasattr(response, "content") and hasattr(response.content, "parts"):
        for part in response.content.parts:
            if hasattr(part, "text") and part.text:
                output_text += part.text + "\n"

    else:
        output_text = str(response)

    return output_text.strip()


if __name__ == "__main__":
    sample_text = """
    Title: خلوت طلبیدن آن ولی از پادشاه جهت دریافتن رنج کنیزک (The Saint Asking the King for Privacy to Understand the Handmaiden's Suffering)
Key Point: The core of the story revolves around a wise physician who understands that physical ailments can have spiritual or emotional roots, and to diagnose them, one must delve deep into the patient's heart and secrets.
Detailed Explanation: In this section of Rumi's Masnavi, a physician is brought to diagnose a sick handmaiden whom no other doctor could cure. The physician astutely realizes her illness is not of the body but of the heart. To uncover the true cause, he asks for complete privacy with the patient. He then gently questions her about her past, her hometown, and her acquaintances, all while keeping his hand on her pulse. The physician compares this process to finding a thorn in one's foot, which requires care and precision, and notes that a "thorn in the heart" is far more difficult to locate.

He systematically names different cities and people, observing the changes in her pulse and complexion. Her pulse remains steady until he mentions the city of Samarkand, at which point it leaps and her face flushes. Through further questioning, he discovers she is in love with a goldsmith from that city. The physician identifies the "illness" as lovesickness, a secret hidden deep within her heart. The key point is that true healing often requires understanding the patient's inner world, secrets, and emotional history, a task that demands wisdom, patience, and a subtle approach rather than just treating physical symptoms. The physician then advises her to keep this secret, stating that keeping secrets helps one achieve their desires more quickly, much like a seed hidden in the earth eventually blossoms into a garden.

    """

    script = generate_youtube_script(sample_text)
    print("\n" + script)


In [ ]:
import json
import base64
from google import genai
from google.genai import types
import re

# Initialize the Gemini client
genai_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"].strip())

def generate_background_images_for_text(text, model="gemini-2.5-pro"):
    """
    Given a text, let the LLM decide how to chunk it and generate a relevant background image for each.
    Returns a JSON structure with text chunks and their corresponding background images.
    
    Uses gemini-2.5-flash-image for actual image generation.
    """
    
    # 1️⃣ Let the LLM decide how to chunk the text optimally
    chunking_prompt = f"""
    Given this text, analyze it and split it into meaningful chunks that would each benefit from a unique background image for a karaoke video.
    
    Text: {text}
    
    Please split this text into logical chunks where each chunk represents a distinct theme, mood, or concept that would benefit from its own background image.
    Return the chunks as a JSON array of strings, where each string is a complete, meaningful portion of the text.
    
    Format: ["chunk1", "chunk2", "chunk3", ...]
    """
    
    chunking_resp = genai_client.models.generate_content(
        model=model,
        contents=chunking_prompt
    )
    
    try:
        # Parse the JSON response to get chunks
        chunks_text = chunking_resp.candidates[0].content.parts[0].text
        # Extract JSON array from the response
        import re
        json_match = re.search(r'\[.*?\]', chunks_text, re.DOTALL)
        if json_match:
            chunks_json = json_match.group(0)
            chunks = json.loads(chunks_json)
        else:
            # Fallback: split by sentences if JSON parsing fails
            chunks = [chunk.strip() for chunk in text.split('.') if chunk.strip()]
    except Exception as e:
        print(f"⚠️ Error parsing chunks, using fallback: {e}")
        # Fallback: split by sentences
        chunks = [chunk.strip() for chunk in text.split('.') if chunk.strip()]

    results = {
        "total_chunks": len(chunks),
        "chunks": []
    }

    # 2️⃣ For each chunk: generate a prompt → then create an image
    for i, chunk in enumerate(chunks, 1):
        print(f"\n🔹 Processing chunk {i}/{len(chunks)}...")

        # Generate a descriptive background prompt
        prompt_resp = genai_client.models.generate_content(
            model=model,
            contents=f"""
Generate a highly detailed, cinematic, ultra-realistic background image for the following text:

{chunk}

Focus on:
- Composition (depth, perspective, and foreground/midground/background layers)
- Lighting (natural or cinematic lighting, soft shadows, reflections)
- Colors (vivid and harmonious color palette)
- Textures (realistic surfaces, details, and depth)
- Mood/Atmosphere corresponding to the text

Style: photorealistic, cinematic, high-resolution, 16:9 aspect ratio
Do not include text in the image.
"""
        )

        image_prompt = prompt_resp.candidates[0].content.parts[0].text
        print(f"🖼️  Image prompt: {image_prompt}")

        # Helper: SVG fallback when binary image generation fails
        def _generate_svg_fallback(prompt: str) -> str:
            svg_prompt = (
                "Create a valid standalone SVG (800x600) that visually represents this description. "
                "Return ONLY the SVG XML from <svg ...> to </svg> with proper xmlns. "
                "Do not include code fences or any extra text.\n\n"
                f"Description: {prompt}"
            )
            svg_resp = genai_client.models.generate_content(
                model=model,
                contents=svg_prompt,
                # Force TEXT modality to ensure SVG comes as text
                config=types.GenerateContentConfig(response_modalities=["TEXT"]),
            )
            raw = svg_resp.candidates[0].content.parts[0].text or ""
            # Extract only the SVG block if model added extra tokens
            import re
            m = re.search(r"<svg[\s\S]*?</svg>", raw)
            if m:
                return m.group(0)
            # As a last resort, synthesize a minimal SVG with the prompt as caption
            safe_text = (prompt or "").replace("<", "&lt;").replace(">", "&gt;")
            return (
                "<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"800\" height=\"600\">"
                "<defs><linearGradient id=\"g\" x1=\"0\" y1=\"0\" x2=\"1\" y2=\"1\">"
                "<stop offset=\"0%\" stop-color=\"#0b1020\"/><stop offset=\"100%\" stop-color=\"#1f2a44\"/>"
                "</linearGradient></defs>"
                "<rect width=\"100%\" height=\"100%\" fill=\"url(#g)\"/>"
                "<circle cx=\"400\" cy=\"300\" r=\"120\" fill=\"#000\"/>"
                "<text x=\"50\" y=\"560\" fill=\"#ddd\" font-size=\"16\" font-family=\"sans-serif\">"
                f"{safe_text[:300]}"
                "</text>"
                "</svg>"
            )

        # Generate the image using gemini-2.5-flash-image
        try:
            max_retries = 2
            last_error = None
            filename = None
            image_base64 = None

            for attempt in range(max_retries + 1):
                try:
                    # Use gemini-2.5-flash-image for actual image generation
                    image_resp = genai_client.models.generate_content(
                        model="gemini-2.5-flash-image",
                        contents=image_prompt,
                        config=types.GenerateContentConfig(
                            response_modalities=["IMAGE"]
                        )
                    )

                    # Extract first available inline image safely
                    image_part = None
                    for cand in getattr(image_resp, "candidates", []) or []:
                        parts = getattr(cand.content, "parts", []) or []
                        for part in parts:
                            inline = getattr(part, "inline_data", None)
                            if inline and getattr(inline, "mime_type", "").startswith("image/"):
                                image_part = inline
                                break
                        if image_part:
                            break

                    if not image_part:
                        raise ValueError("Model returned no inline image data")

                    image_bytes = base64.b64decode(image_part.data)

                    # Basic validation of image bytes (PNG/JPEG headers)
                    is_png = image_bytes.startswith(b"\x89PNG\r\n\x1a\n")
                    is_jpg = image_bytes.startswith(b"\xff\xd8")
                    if not (is_png or is_jpg):
                        raise ValueError("Generated bytes are not valid PNG/JPEG")

                    mime = getattr(image_part, "mime_type", "image/png")
                    ext = ".png" if (is_png or mime.endswith("png")) else ".jpg"

                    # Convert image bytes to base64 for JSON storage
                    image_base64 = base64.b64encode(image_bytes).decode('utf-8')

                    # Save locally
                    filename = f"background_{i}{ext}"
                    with open(filename, "wb") as f:
                        f.write(image_bytes)
                    print(f"✅ Saved {filename}")
                    break  # success

                except Exception as inner_e:
                    last_error = inner_e
                    if attempt < max_retries:
                        print(f"↩️ Retry {attempt+1}/{max_retries} for chunk {i} due to: {inner_e}")
                    else:
                        # On final failure, try SVG fallback
                        try:
                            svg_text = _generate_svg_fallback(image_prompt)
                            if not svg_text.strip().startswith("<svg"):
                                raise ValueError("SVG fallback did not return SVG XML")
                            filename = f"background_{i}.svg"
                            with open(filename, "w") as f:
                                f.write(svg_text)
                            image_base64 = base64.b64encode(svg_text.encode("utf-8")).decode("utf-8")
                            print(f"✅ Saved {filename} (SVG fallback)")
                        except Exception as svg_e:
                            raise ValueError(f"SVG fallback failed: {svg_e}")

            # Add to results (success path)
            chunk_data = {
                "chunk_id": i,
                "text": chunk,
                "image_prompt": image_prompt,
                "image_filename": filename,
                "image_base64": image_base64
            }
            results["chunks"].append(chunk_data)

        except Exception as e:
            print(f"❌ Error generating image for chunk {i}: {str(e)}")
            # Add chunk without image
            chunk_data = {
                "chunk_id": i,
                "text": chunk,
                "image_prompt": image_prompt,
                "image_filename": None,
                "image_base64": None,
                "error": str(e)
            }
            results["chunks"].append(chunk_data)

    return results


# Test the function with sample text
cleaned_script = re.sub(r'\([^)]*\)', '', script)
sample_text = cleaned_script


# Generate background images and return as JSON
background_results = generate_background_images_for_text(sample_text)

# Save results to JSON file
with open("background_images.json", "w") as f:
    json.dump(background_results, f, indent=2)

print(f"\n📄 JSON results saved to background_images.json")
print(f"📊 Generated {background_results['total_chunks']} background images")
print(f"🎨 Each chunk includes text, image prompt, filename, and base64 encoded image")


In [ ]:
import base64
import wave
from google import genai
from google.genai import types
from pydub import AudioSegment

def save_wav(filename, pcm_bytes, channels=1, rate=24000, sample_width=2):
    """Save raw PCM bytes into a WAV file with proper headers."""
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm_bytes)

# --- Gemini client ---
client = genai.Client()

# --- Style / Tone prompt ---
style_prompt = """
Speak like an experienced academic science teacher presenting on YouTube.
Use a confident, clear, and engaging tone — professional but approachable.
Emphasize key ideas naturally, and use slight pauses to help the listener follow complex explanations.
Avoid robotic or monotonous delivery; sound human, thoughtful, and enthusiastic about teaching.
"""

# --- Arabic content (separated cleanly) ---
arabic_text ="""
What if the most powerful medicine in the world wasn't a pill or a procedure, but a conversation? What if the root of a mysterious illness wasn't in the body, but hidden deep within the heart?

This is the story of a wise physician from the Masnavi, a timeless collection of tales by the poet Rumi. It’s a story that reveals a profound truth about healing that we still grapple with today.

Our story begins in a royal court. A king had a favorite handmaiden who suddenly fell gravely ill. She wasted away, growing pale and weak, and no one could figure out why. The king summoned the most famous doctors in the land. They tried every remedy, every potion, every treatment they knew, but nothing worked. The handmaiden only grew worse.

Finally, a new physician arrived. He was a man of great wisdom and insight. After observing the handmaiden, he suspected something the other doctors had missed. He realized her sickness might not be in her body at all. It was a sickness of the soul.

To find the cure, he knew he had to uncover a secret.

He went to the king and made a strange request. He asked for complete privacy with the patient. No one else was to be in the room. The king, desperate, agreed.

Alone with the handmaiden, the physician sat beside her and gently took her hand, placing his fingers on her pulse. He didn't ask about her physical symptoms. Instead, he began to ask about her life. He asked about her hometown, the people she knew, the places she had been.

Rumi compares this delicate process to removing a thorn from someone’s foot. You have to be careful, precise, and gentle. But, Rumi notes, a thorn in the heart is a thousand times harder to find.

The physician began to list the names of different cities, one by one, all while watching her and feeling the rhythm of her pulse. Her pulse remained steady, calm, and weak. But then, he mentioned the city of Samarkand.

Instantly, her pulse leaped. A flush of color returned to her pale cheeks.

The physician knew he was close. He now began to name the different districts and streets of Samarkand. He then named the various tradesmen and artisans of the city. Finally, he mentioned a certain young goldsmith.

At the sound of the goldsmith’s name, the handmaiden’s pulse went wild, and her face turned bright red. The secret was revealed. The physician had found the thorn in her heart. Her mysterious illness was not an illness at all. It was lovesickness. She was deeply in love with a goldsmith from her hometown, a secret she had locked away so deep inside that it was physically consuming her.

The diagnosis was complete. But the physician’s wisdom didn't stop there. He gave her some surprising advice. He told her to keep this secret, to guard it carefully.

Why? Because, as he explained, when you keep a secret desire hidden, it’s like planting a seed in the dark earth. If you keep digging it up to see if it’s growing, it will die. But if you let it rest, hidden and protected, it will eventually sprout and blossom into a beautiful garden. Keeping a secret, he said, helps you achieve your goal more quickly.

This ancient story teaches us a powerful lesson. True healing isn't just about treating symptoms. It's about understanding the whole person—their history, their emotions, and the secrets they carry in their hearts. It reminds us that our minds and bodies are deeply connected, and sometimes, the most profound cure is found not in a medicine bottle, but in wisdom, patience, and a willingness to listen to the silent stories the heart has to tell.




"""

# --- Request TTS ---
response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=[
        style_prompt,
        arabic_text
    ],
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name='iapetus',  # Try 'Kore', 'Wavenet', etc.
                )
            )
        ),
    )
)

# --- Extract audio (decode base64 PCM) ---
audio_data = response.candidates[0].content.parts[0].inline_data.data
# Convert PCM to MP3 using pydub
audio = AudioSegment(
    data=audio_data,
    sample_width=2,     # 16-bit PCM
    frame_rate=24000,   # same as TTS output
    channels=1
)
audio.export("youtube_audio.mp3", format="mp3")
print("🎶 Saved audio as youtube.mp3")



In [ ]:
#Audio generation is working well.

# forced alignment tool to get timestamps for audio and text.
# generate video

In [ ]:
import whisper

# Load the model (small/base/medium/large)
model = whisper.load_model("base")

# Transcribe your Gemini audio with word-level timestamps
result = model.transcribe("arabic_audio.mp3", word_timestamps=True)

# Print the full transcription to see what Whisper got
print("Full transcription:")
print(result["text"])
print(f"\nNumber of segments: {len(result['segments'])}")

# result["segments"] contains word timestamps
words = []
for segment in result["segments"]:
    print(f"Segment: {segment['text']} (start: {segment['start']:.2f}, end: {segment['end']:.2f})")
    for w in segment["words"]:
        words.append({
            "word": w["word"],
            "start": w["start"],
            "end": w["end"]
        })

print(f"\nTotal words found: {len(words)}")
print("First 10 words:")
print(words[:10])
print("Last 10 words:")
print(words[-10:])


In [ ]:
# Generate JSON configuration for Remotion karaoke workflow
import json

# Create JSON configuration for karaoke video generation
karaoke_config = {
    "audio": "arabic_audio.mp3",
    "words": words,  # Use the words from Whisper transcription
    "output": "youtube_karaoke",
    "words_per_line": 5,
    "background_image": "generated_image.png"
}

# Save configuration to JSON file
config_filename = "youtube_karaoke_config.json"
with open(config_filename, 'w') as f:
    json.dump(karaoke_config, f, indent=2)

print(f"✅ JSON configuration created: {config_filename}")
print(f"📄 Found {len(words)} words for karaoke video")
print(f"🎵 Audio file: Malfuzat.mp3")
print(f"📝 Output: Malfuzat_karaoke.mp4")
print(f"📏 Words per line: 5")
print(f"🖼️ Background image: Generated Image.png")

print("\n🎬 To generate karaoke video:")
print(f"cd ./remotion-karaoke")
print(f"python3 karaoke_workflow.py ../{config_filename}")


In [ ]:
# Create an image using Nano Banana. 
"""
with the following prompt:
Text: "Patience is a great quality. He who shows patience and does not talk angrily, is the one who when he speaks, it is not he who speaks, it is God who makes him speak. The members of the Jamaat should show patience and they should not be irritated at the opposition of others nor should they pay back in the same coins; they should not abuse the people who abuse them. He who takes me as a liar, is not expected to show me respect.     

I want the background image for a karaoke style video about the theme in the text.
"""

In [ ]:
# Build JSON for ScienceVideo from existing Whisper timestamps
# - Uses `words` list created by your Whisper cell (Cell 5)
# - Uses the audio file saved by the TTS cell (defaults to 'Blackholes.mp3')
# - Writes out `science_karaoke.json`

import json

# If you used a different filename in your TTS cell, change this:
AUDIO_FILE = "Blackholes.mp3"
OUTPUT_JSON = "science_karaoke.json"

if 'words' not in globals() or not isinstance(words, list) or not words:
    raise RuntimeError("Expected Whisper variable `words` to exist and be a non-empty list. Run the Whisper cell first.")

payload = {
    "audio_file": AUDIO_FILE,
    "words": words,
}

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"✅ Wrote {len(words)} words to {OUTPUT_JSON}")
print(f"🎵 audio_file: {AUDIO_FILE}")



In [ ]:
# Create a mapping JSON that aligns background images with word timestamps
# Using sequential word matching to avoid overlaps
import json
import re

def align_chunks_with_timestamps(transcription_data, chunks_data):
    """
    Aligns chunks with transcription data using sequential word matching.
    Returns a list of background switches with precise timestamps.
    """
    words = transcription_data['words']
    chunks = chunks_data['chunks']
    
    # A cursor to track our position in the main 'words' list.
    # This avoids re-scanning from the beginning for each chunk.
    word_cursor = 0
    background_switches = []

    for chunk in chunks:
        # Normalize the chunk's text to prepare it for matching.
        # This removes punctuation and handles different apostrophe types.
        text_for_matching = chunk['text'].replace("'", "'").strip()
        words_in_chunk = [re.sub(r'[^\w\']', '', word).lower() for word in text_for_matching.split()]
        
        # Find the starting word of the chunk in the transcription list
        start_word_index = -1
        for i in range(word_cursor, len(words)):
            # Normalize the word from the transcription list for comparison
            current_word = re.sub(r'[^\w\']', '', words[i]['word']).lower()
            if words_in_chunk and current_word == words_in_chunk[0]:
                start_word_index = i
                break
        
        if start_word_index == -1:
            print(f"Warning: Could not find a starting match for Chunk {chunk['chunk_id']}.")
            continue

        # Determine the start time from the found word
        start_time = words[start_word_index]['start']
        
        # The end word index is calculated based on the number of words in the chunk
        end_word_index = start_word_index + len(words_in_chunk) - 1
        
        # Ensure the end index is within the bounds of the words list
        if end_word_index >= len(words):
            end_word_index = len(words) - 1
            
        # Get the end time and the final word of the chunk
        end_time = words[end_word_index]['end']
        last_word_in_transcript = words[end_word_index]['word']
        
        # Clean the final word by removing any trailing punctuation for display
        last_word_cleaned = re.sub(r'[^\w\s\']', '', last_word_in_transcript).strip()
        
        # Format and print the final output for the current chunk
        print(f"Chunk {chunk['chunk_id']}: {start_time:.2f}s - {end_time:.2f}s (ends with \"{last_word_cleaned}\")")
        
        # Create background switch entry
        background_switch = {
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "image_filename": chunk["image_filename"],
            "start_time": start_time,
            "end_time": end_time,
            "duration": end_time - start_time
        }
        background_switches.append(background_switch)
        
        # Move the cursor forward to optimize the next search
        word_cursor = end_word_index + 1

    return background_switches

# Load the existing data
with open('background_images.json', 'r') as f:
    background_data = json.load(f)

with open('science_karaoke.json', 'r') as f:
    karaoke_data = json.load(f)

# Create the mapping using sequential word matching
background_switches = align_chunks_with_timestamps(karaoke_data, background_data)

# Create the final mapping
background_timestamps = {
    "audio_file": karaoke_data["audio_file"],
    "total_chunks": background_data["total_chunks"],
    "total_duration": karaoke_data['words'][-1]['end'],
    "background_switches": background_switches
}

# Save the mapping
output_file = "background_timestamps.json"
with open(output_file, 'w') as f:
    json.dump(background_timestamps, f, indent=2)

print(f"\n📄 Background timestamps saved to: {output_file}")
print(f"🎬 Total background switches: {len(background_timestamps['background_switches'])}")
print(f"⏱️  Total duration: {background_timestamps['total_duration']:.2f}s")
print(f"📊 Sequential word matching - no overlaps, precise timestamps")
